# 01 — Bytes, C Memory, and Ownership Contracts

**Modules:** `00.04`, `00.05`, `00.06`  
**Prerequisite:** none; this notebook defines the byte contract reused by the whole Transistor spine.  
**Emits:** `fromthetransistor/01-byte-contract.json`

A C value, its mathematical interpretation, and its stored bytes are related but not identical. We make width, signed interpretation, byte order, pointer scaling, packed layout, and ownership explicit before later notebooks treat bytes as UART frames, instructions, pages, packets, or flash data. Predict each representation before running its assertion.

In [ ]:
from pathlib import Path
import json
import os
import struct

ARTIFACT_ROOT = Path(os.environ['COURSE_NOTEBOOK_ARTIFACTS'])
WORD_BITS = 16
BYTEORDER = 'little'
WORD_MASK = (1 << WORD_BITS) - 1

def as_u16(value):
    return value & WORD_MASK

def as_i16(value):
    value &= WORD_MASK
    return value - (1 << WORD_BITS) if value & (1 << (WORD_BITS - 1)) else value

def word_bytes(value):
    return as_u16(value).to_bytes(WORD_BITS // 8, BYTEORDER)

def word_from_bytes(data):
    if len(data) != WORD_BITS // 8:
        raise ValueError('one word requires exactly two bytes')
    return int.from_bytes(data, BYTEORDER)

assert as_u16(-42) == 65494
assert as_i16(as_u16(-42)) == -42
assert word_bytes(-42) == bytes([0xD6, 0xFF])
assert word_from_bytes(word_bytes(-42)) == as_u16(-42)


## Worked analysis

Pointer arithmetic scales by the pointed-to element size; it is not ordinary byte addition. A packed wire or disk layout also differs from a compiler's native ABI layout. The tiny `Region` below makes an ownership transition visible so a use-after-free becomes a deterministic failure instead of accidental behavior.

In [ ]:
def element_address(base, index, element_size):
    if index < 0 or element_size <= 0:
        raise ValueError('invalid index or element size')
    return base + index * element_size

class Region:
    def __init__(self, data):
        self._data = bytearray(data)
        self._alive = True
    def read(self, index):
        if not self._alive:
            raise RuntimeError('use after free')
        return self._data[index]
    def free(self):
        if not self._alive:
            raise RuntimeError('double free')
        self._alive = False

base = 0x1000
addresses = [element_address(base, i, 4) for i in range(4)]
packed_header = struct.pack('<IhB', 0x12345678, -7, 3)
assert addresses == [0x1000, 0x1004, 0x1008, 0x100C]
assert len(packed_header) == 7
assert packed_header[:4] == bytes([0x78, 0x56, 0x34, 0x12])
region = Region(b'ABC')
assert region.read(1) == ord('B')
region.free()
try:
    region.read(0)
    raise AssertionError('dead region was readable')
except RuntimeError as error:
    assert str(error) == 'use after free'

artifact = {
    'schema_version': 1,
    'word_bits': WORD_BITS,
    'byteorder': BYTEORDER,
    'signed_encoding': 'twos-complement',
    'pointer_rule': 'base + index * element_size',
    'wire_layout': '<IhB',
    'known_word_hex': word_bytes(-42).hex(),
}
target = ARTIFACT_ROOT / 'fromthetransistor/01-byte-contract.json'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text(json.dumps(artifact, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print('byte contract:', artifact)


## Exercise and falsifiers

1. Predict the two bytes for signed `-2`, then check them.
2. Find the address of element 7 in an array of 32-bit values at `0x2000`.
3. Change the byte order deliberately and explain which later interfaces would break.
4. Extend `Region` with a bounds check and a single-owner transfer operation.

The self-check cell records known answers but does not prove that native C ABI padding matches the explicit packed contract.

In [ ]:
exercise_minus_two = word_bytes(-2)
exercise_address = element_address(0x2000, 7, 4)
assert exercise_minus_two == bytes([0xFE, 0xFF])
assert exercise_address == 0x201C
assert int.from_bytes(exercise_minus_two, 'big') != as_u16(-2)
print('self-checks passed; next notebook must consume the emitted byte contract')
